In [26]:
import os
import json
import numpy as np
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
 
load_dotenv()

True

In [27]:
json_dir = './json/content'
section_names = [x.replace('.json','') for x in os.listdir(json_dir) if '.json' in x]

docs = []
for i,section_name in enumerate(section_names):
    with open(f'{json_dir}/{section_name}.json', 'r') as f:
        # Parsing the JSON file into a Python dictionary
        content_list = json.load(f)
        for j,content in enumerate(content_list):
            text_content = f"Section: {section_name}\n\nTitle: {content['title']}\n\nContent: {content['content']}"
            metadata = {
                "section": section_name,
                "title": content['title'],
                "doc_id": f"section{i}-title{j}"
            }
            doc = Document(
                                page_content=text_content,
                                metadata=metadata
                            )
            docs.append(doc)

In [28]:
doc_lens = [len(doc.page_content) for doc in docs]
np.mean(doc_lens)

np.float64(5753.985)

In [32]:
recursive_splitter = RecursiveCharacterTextSplitter(separators=['\n\n','\n','. ','?','; ',':',', '], 
                                                    chunk_size=300,
                                                    chunk_overlap=0)

In [33]:
chunks = recursive_splitter.split_documents(docs)
print(len(chunks))

5455


In [34]:
# chunks

In [40]:
for i, chunk in enumerate(chunks[1001:1005]):
    print(f"\nChunk {i}: len: {len(chunk.page_content)}")
    print(chunk.page_content)
    print(chunk.metadata)


Chunk 0: len: 203
To access your Binance Wallet, you must have at least 2 wallet key-shares. In the unfortunate event that a key-share is compromised, the remaining ones are still securely encrypted to protect your assets
{'section': 'Binance Wallet', 'title': 'What Is a Keyless Wallet and Why Do I Need to Back It Up?', 'doc_id': 'section4-title7'}

Chunk 1: len: 125
. Therefore, to prevent the loss of assets, you should back up your Binance Wallet and store your recovery password securely.
{'section': 'Binance Wallet', 'title': 'What Is a Keyless Wallet and Why Do I Need to Back It Up?', 'doc_id': 'section4-title7'}

Chunk 2: len: 94
However, it's crucial to understand that deleting the App could result in the loss of Share 2.
{'section': 'Binance Wallet', 'title': 'What Is a Keyless Wallet and Why Do I Need to Back It Up?', 'doc_id': 'section4-title7'}

Chunk 3: len: 232
This underscores the importance of the backup process for Share 1 and Share 3. Without backups for these key-shar

In [41]:
"""Create and persist ChromDB vector store"""
persist_directory = "db/Chroma_db"
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Create vector store Chroma-DB
print("-- Creating Vector Store")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory,
    collection_metadata={"hnsw:space":"cosine"}
)
print("-- Finish Creating Vector Store")
print(f"Vector Store created and saved to {persist_directory}")

-- Creating Vector Store
-- Finish Creating Vector Store
Vector Store created and saved to db/Chroma_db
